## 1. Setup

In [2]:
import sys, os
from pathlib import Path
from typing import List, Dict
from dotenv import load_dotenv

# Setup paths
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
os.chdir(project_root)
sys.path.insert(0, str(project_root))
load_dotenv()

from src.services.llm_services import load_config, get_llm, get_text_embeddings

config = load_config("src/config/config.yaml")
print(f"LLM: {config['llm_model']}")
print(f"Embeddings: {config['text_emb_model']}")

LLM: llama-3.1-8b-instant
Embeddings: sentence-transformers/all-MiniLM-L6-v2


## 2. Connect to Weaviate

In [3]:
import weaviate

cloud_config = config["weaviate"]["cloud"]
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=cloud_config["cluster_url"],
    auth_credentials=weaviate.auth.AuthApiKey(cloud_config["api_key"]),
)

collection_name = config["weaviate"]["schema"]["class_name"]
collection = client.collections.get(collection_name)
count = collection.aggregate.over_all(total_count=True).total_count

print(f"Connected! Collection: {collection_name}, Chunks: {count}")

Connected! Collection: UberReportChunk, Chunks: 464


## 3. Load Components

In [4]:
from sentence_transformers import CrossEncoder

embeddings = get_text_embeddings(config)
reranker = CrossEncoder(config["retrieval"]["reranker_model"], max_length=512)
llm = get_llm(config)

print("Components loaded!")

c:\Users\viraj\Zuu\Ledger_mind\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\viraj\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back t

Components loaded!


## 4. Search Functions

In [5]:
def hybrid_search(query: str, top_k: int = 20, alpha: float = 0.6) -> List[Dict]:
    """Hybrid search: dense + BM25. alpha=1 is pure vector, alpha=0 is pure BM25."""
    query_vector = embeddings.embed_query(query)
    response = collection.query.hybrid(
        query=query, vector=query_vector, alpha=alpha, limit=top_k,
        return_metadata=["score"], return_properties=["content", "chunk_index", "source"]
    )
    return [{"content": o.properties["content"], "chunk_index": o.properties["chunk_index"],
             "source": o.properties["source"], "score": o.metadata.score or 0} for o in response.objects]

def dense_search(query: str, top_k: int = 20) -> List[Dict]:
    """Pure vector search."""
    query_vector = embeddings.embed_query(query)
    response = collection.query.near_vector(
        near_vector=query_vector, limit=top_k,
        return_metadata=["distance"], return_properties=["content", "chunk_index", "source"]
    )
    return [{"content": o.properties["content"], "chunk_index": o.properties["chunk_index"],
             "source": o.properties["source"], "distance": o.metadata.distance or 1} for o in response.objects]

def bm25_search(query: str, top_k: int = 20) -> List[Dict]:
    """Pure BM25 keyword search."""
    response = collection.query.bm25(
        query=query, limit=top_k,
        return_metadata=["score"], return_properties=["content", "chunk_index", "source"]
    )
    return [{"content": o.properties["content"], "chunk_index": o.properties["chunk_index"],
             "source": o.properties["source"], "score": o.metadata.score or 0} for o in response.objects]

print("Search functions ready!")

Search functions ready!


## 5. Rerank Function

In [6]:
def rerank(query: str, docs: List[Dict], top_n: int = 3) -> List[Dict]:
    """Rerank using cross-encoder."""
    if not docs:
        return []
    pairs = [(query, d["content"]) for d in docs]
    scores = reranker.predict(pairs, show_progress_bar=False)
    for d, s in zip(docs, scores):
        d["rerank_score"] = float(s)
    return sorted(docs, key=lambda x: x["rerank_score"], reverse=True)[:top_n]

print("Rerank function ready!")

Rerank function ready!


## 6. RAG Query

In [7]:
def rag_query(question: str, top_k: int = 20, top_n: int = 3, verbose: bool = True) -> Dict:
    """Full RAG: Hybrid Retrieval -> Rerank -> LLM."""
    
    # Retrieve
    if verbose: print(f"[1] Retrieving top {top_k}...")
    docs = hybrid_search(question, top_k=top_k)
    if not docs:
        return {"answer": "No relevant info found.", "sources": []}
    
    # Rerank
    if verbose: print(f"[2] Reranking to top {top_n}...")
    top_docs = rerank(question, docs, top_n=top_n)
    if verbose:
        for i, d in enumerate(top_docs, 1):
            print(f"    {i}. Chunk {d['chunk_index']} (score: {d['rerank_score']:.3f})")
    
    # Build context
    context = "\n\n".join([f"[Doc {i+1}]\n{d['content']}" for i, d in enumerate(top_docs)])
    
    # Generate
    if verbose: print(f"[3] Generating...")
    prompt = f"""Use the context to answer. Be concise.

Context:
{context}

Question: {question}

Answer:"""
    
    resp = llm.invoke(prompt)
    answer = resp.content if hasattr(resp, "content") else str(resp)
    
    if verbose: print("[4] Done!\n")
    
    return {
        "answer": answer,
        "sources": [{"chunk": d["chunk_index"], "score": d["rerank_score"]} for d in top_docs]
    }

print("rag_query ready!")

rag_query ready!


## 7. Test It!

In [8]:
result = rag_query("What is Uber's mission?")
print("="*50)
print(result["answer"])

[1] Retrieving top 20...
[2] Reranking to top 3...
    1. Chunk 0 (score: 4.000)
    2. Chunk 10 (score: 0.064)
    3. Chunk 205 (score: -0.059)
[3] Generating...
[4] Done!

Uber's mission is to "reimagine the way the world moves for the better."


In [9]:
result = rag_query("What was Uber's revenue in 2024?")
print("="*50)
print(result["answer"])

[1] Retrieving top 20...
[2] Reranking to top 3...
    1. Chunk 235 (score: 5.724)
    2. Chunk 222 (score: 3.460)
    3. Chunk 352 (score: 2.387)
[3] Generating...
[4] Done!

$43,978 million.


## 8. Compare Methods

In [10]:
def compare(query: str, k: int = 5):
    print(f"Query: '{query}'\n")
    d = [r["chunk_index"] for r in dense_search(query, k)]
    b = [r["chunk_index"] for r in bm25_search(query, k)]
    h = [r["chunk_index"] for r in hybrid_search(query, k)]
    print(f"Dense:  {d}")
    print(f"BM25:   {b}")
    print(f"Hybrid: {h}")

compare("Form 10-K filing")

Query: 'Form 10-K filing'

Dense:  [453, 9, 210, 33, 261]
BM25:   [453, 459, 460, 456, 215]
Hybrid: [453, 9, 210, 33, 261]


## 9. Cleanup

In [ ]:
client.close()
print("Done!")